# RAG with Llama Stack

In this notebook we query the model using **Retrieval-Augmented Generation (RAG)**.
The documents were already ingested into a Milvus vector store by the pipeline in `rag_pipeline.py`.

Instead of using the OpenAI chat completions API, we use the **Llama Stack Responses API** which
lets us attach a `file_search` tool backed by our vector store. When the model receives a question,
Llama Stack automatically retrieves the most relevant document chunks and includes them in the context.

We also trace every request to MLflow so we can inspect inputs, retrieved chunks, and responses.

## Install and import dependencies

We use the `llama-stack-client` package to talk to our Llama Stack server, and `mlflow` for tracing.

In [ ]:
!pip install -q "llama-stack-client>=0.7" mlflow

In [ ]:
import mlflow
from llama_stack_client import LlamaStackClient
import os

import warnings
warnings.filterwarnings("ignore")

## Configuration

First we configure MLflow — same setup as in `1_mflow_request.ipynb`.
Fill in your `MLFLOW_TRACKING_TOKEN` if required by your deployment.

Then we point the Llama Stack client at the server and set the `VECTOR_STORE_ID`
you noted from the pipeline run output.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MLFLOW Configuration
# ─────────────────────────────────────────────────────────────────

# MLflow server
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")

# MLflow names (change if you want to)
EXPERIMENT_NAME = "hospital-helpdesk"
PROMPT_NAME = "ai-hospital-helpdesk"

# Connect to MLflow
os.environ["MLFLOW_WORKSPACE"] = "hospital-helpdesk"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

experiment_id = mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
# Llama Stack server
LLAMA_STACK_URL = os.getenv("LLAMA_STACK_URL", "http://lsd-genai-playground-service.hospital-helpdesk.svc.cluster.local:8321")
LLAMA_STACK_MODEL = os.getenv("LLAMA_STACK_MODEL", "vllm-inference-1/redhataillama-32-3b-instruct-q")

# Vector store ID — copy this from the pipeline run output in rag_pipeline.py,
# or leave empty to automatically use the first available vector store.
VECTOR_STORE_ID = ""

# Lower temperature → more deterministic tool calls and factual answers (recommended for RAG)
# Range: 0.0 (fully deterministic) – 1.0 (creative/varied)
TEMPERATURE = 0.1

client = LlamaStackClient(base_url=LLAMA_STACK_URL, timeout=300)

# Auto-discover vector store if not specified
if not VECTOR_STORE_ID:
    stores = client.vector_stores.list()
    if not stores.data:
        raise RuntimeError("No vector stores found. Run rag_pipeline.py first to ingest documents.")
    VECTOR_STORE_ID = stores.data[0].id
    print(f"Auto-selected vector store: {VECTOR_STORE_ID} (name: {stores.data[0].name})")
else:
    print(f"Using configured vector store: {VECTOR_STORE_ID}")

## Inspect the vector store

Before sending requests, let's peek at what's actually inside the vector store — which files were ingested and a sample of the chunks the model will be able to retrieve.

In [ ]:
# List files ingested into the vector store
files = client.vector_stores.files.list(vector_store_id=VECTOR_STORE_ID)
print(f"Files in vector store '{VECTOR_STORE_ID}':")
for f in files.data:
    print(f"  - {f.id}  (status: {f.status})")

# Sample chunks via a broad search
print("\nSample chunks:")
results = client.vector_stores.search(
    vector_store_id=VECTOR_STORE_ID,
    query="hospital",
    max_num_results=3,
)
for i, item in enumerate(results.data, 1):
    # Each item has a list of content parts; grab the text
    text = next((p.text for p in item.content if p.type == "text"), "")
    preview = text[:400].replace("\n", " ")
    print(f"\n[Chunk {i}] score={item.score:.3f}")
    print(f"  {preview}{'...' if len(text) > 400 else ''}")

## Load the system prompt from MLflow

We load the same system prompt we registered in MLflow during chapter 3, so the model keeps its
hospital helpdesk persona when answering RAG-enhanced queries.

In [ ]:
sys_prompt_mlflow = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@latest")
sys_prompt_plaintext = next(m["content"] for m in sys_prompt_mlflow.format() if m["role"] == "system")
sys_prompt_plaintext

## Send RAG requests

We use the **Llama Stack Responses API** (`client.responses.create`) with a `file_search` tool.
When a request comes in, Llama Stack:
1. Encodes the question into an embedding
2. Searches the vector store for the most similar document chunks
3. Injects those chunks into the model context
4. Returns the model's grounded answer

The `@mlflow.trace` decorator records the full call — input question and model answer — as a
trace in MLflow, just like the autolog in `1_mflow_request.ipynb`.

In [ ]:
import re
import mlflow

def send_rag_request(message: str) -> dict:
    """Send a RAG request and return the answer and retrieved chunks."""
    tools = [{"type": "file_search", "vector_store_ids": [VECTOR_STORE_ID]}]

    with mlflow.start_span(name=message, span_type="CHAIN") as root_span:
        root_span.set_inputs(message)

        stream = client.responses.create(
            model=LLAMA_STACK_MODEL,
            input=message,
            instructions=sys_prompt_plaintext,
            tools=tools,
            temperature=TEMPERATURE,
            stream=True,
        )

        full_text = ""
        call_index = 0
        all_chunks = []

        for event in stream:
            etype = getattr(event, "type", "")

            if etype == "response.output_text.delta":
                full_text += event.delta

            elif etype == "response.output_item.done":
                item = getattr(event, "item", None)
                if item and getattr(item, "type", "") == "file_search_call":
                    call_index += 1
                    results = getattr(item, "results", []) or []
                    chunks = [
                        {
                            "text": getattr(r, "text", ""),
                            "score": getattr(r, "score", None),
                            "filename": getattr(r, "filename", None),
                        }
                        for r in results
                    ]
                    seen = {c["text"] for c in all_chunks}
                    for c in chunks:
                        if c["text"] not in seen:
                            all_chunks.append(c)
                            seen.add(c["text"])
                    with mlflow.start_span(name=f"file_search #{call_index}", span_type="RETRIEVER") as span:
                        span.set_inputs({"query": getattr(item, "query", message), "vector_store_ids": [VECTOR_STORE_ID]})
                        span.set_outputs({"chunks": chunks, "num_chunks": len(chunks)})

        full_text = re.sub(r"<\|file-[a-f0-9]+\|", "", full_text).strip()
        root_span.set_outputs(full_text)

    return {"answer": full_text, "chunks": all_chunks}


In [ ]:
messages = [
    "I forgot my password and am locked out of my account. How do I reset it?",
    "Where is the Biomedical Engineering department located?",
    "What is the burn-in period for new critical medical equipment?",
]

In [ ]:
width = 100
SHOW_CHUNKS = False

for message in messages:
    print(f"\n{'━' * width}")
    print(f"  Q: {message}")
    print(f"{'━' * width}")
    
    result = send_rag_request(message)
    answer = result["answer"]
    chunks = result["chunks"]

    if chunks and SHOW_CHUNKS:
        print(f"  📄 Retrieved {len(chunks)} chunk(s):")
        for j, chunk in enumerate(chunks, 1):
            fname = chunk["filename"] or "unknown"
            score = f"  score={chunk['score']:.3f}" if chunk["score"] is not None else ""
            preview = chunk["text"].replace("\n", " ").strip()
            if len(preview) > 300:
                preview = preview[:297] + "..."
            print(f"  ┌─ [{j}] {fname}{score}")
            print(f"  │  {preview}")
        print()

    print(f"  💬 Answer:")
    for line in answer.splitlines():
        print(f"  {line}")
    print(f"{'━' * width}\n")